# Kaggle Player Scores / Transfermarkt Data Inventory

Goal: Explore the Kaggle dataset `davidcariboo/player-scores` as a possible player-focused football data source.

This notebook inventories all raw CSV files and briefly inspects important player-related tables.

No cleaning, feature engineering, SQL import, or modeling in this notebook.

In [1]:
from pathlib import Path
import json
from datetime import datetime, timezone

import pandas as pd

RAW_DIR = Path("../data/raw/kaggle_player_scores")
RAW_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(RAW_DIR.glob("*.csv"))

csv_files

[PosixPath('../data/raw/kaggle_player_scores/appearances.csv'),
 PosixPath('../data/raw/kaggle_player_scores/club_games.csv'),
 PosixPath('../data/raw/kaggle_player_scores/clubs.csv'),
 PosixPath('../data/raw/kaggle_player_scores/competitions.csv'),
 PosixPath('../data/raw/kaggle_player_scores/countries.csv'),
 PosixPath('../data/raw/kaggle_player_scores/game_events.csv'),
 PosixPath('../data/raw/kaggle_player_scores/game_lineups.csv'),
 PosixPath('../data/raw/kaggle_player_scores/games.csv'),
 PosixPath('../data/raw/kaggle_player_scores/national_teams.csv'),
 PosixPath('../data/raw/kaggle_player_scores/player_valuations.csv'),
 PosixPath('../data/raw/kaggle_player_scores/players.csv'),
 PosixPath('../data/raw/kaggle_player_scores/transfers.csv')]

In [2]:
def count_csv_rows(path: Path) -> int:
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return max(sum(1 for _ in f) - 1, 0)


inventory = []

for path in csv_files:
    sample = pd.read_csv(path, nrows=5)
    
    inventory.append({
        "file": path.name,
        "rows": count_csv_rows(path),
        "columns": len(sample.columns),
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2),
        "column_names": list(sample.columns),
    })

inventory_df = pd.DataFrame(inventory).sort_values("file")

inventory_df

,file,rows,columns,size_mb,column_names
0,appearances.csv,1862208,13,139.77,"[appearance_id, game_id, player_id, player_clu..."
1,club_games.csv,173966,11,10.29,"[game_id, club_id, own_goals, own_position, ow..."
2,clubs.csv,796,17,0.18,"[club_id, club_code, name, domestic_competitio..."
3,competitions.csv,67,11,0.01,"[competition_id, competition_code, name, sub_t..."
4,countries.csv,118,8,0.01,"[country_id, country_name, country_code, confe..."
5,game_events.csv,1242945,11,145.83,"[game_event_id, date, game_id, minute, type, c..."
6,game_lineups.csv,3049833,10,322.34,"[game_lineups_id, date, game_id, player_id, cl..."
7,games.csv,86983,23,24.51,"[game_id, competition_id, season, round, date,..."
8,national_teams.csv,118,17,0.02,"[national_team_id, name, team_code, country_id..."
9,player_valuations.csv,616377,6,28.63,"[player_id, date, market_value_in_eur, current..."


In [6]:
inventory_df[["file", "rows", "columns", "size_mb"]]

,file,rows,columns,size_mb
0,appearances.csv,1862208,13,139.77
1,club_games.csv,173966,11,10.29
2,clubs.csv,796,17,0.18
3,competitions.csv,67,11,0.01
4,countries.csv,118,8,0.01
5,game_events.csv,1242945,11,145.83
6,game_lineups.csv,3049833,10,322.34
7,games.csv,86983,23,24.51
8,national_teams.csv,118,17,0.02
9,player_valuations.csv,616377,6,28.63


In [7]:
for _, row in inventory_df.iterrows():
    print(f"\n--- {row['file']} ---")
    for column in row["column_names"]:
        print(column)


--- appearances.csv ---
appearance_id
game_id
player_id
player_club_id
player_current_club_id
date
player_name
competition_id
yellow_cards
red_cards
goals
assists
minutes_played

--- club_games.csv ---
game_id
club_id
own_goals
own_position
own_manager_name
opponent_id
opponent_goals
opponent_position
opponent_manager_name
hosting
is_win

--- clubs.csv ---
club_id
club_code
name
domestic_competition_id
total_market_value
squad_size
average_age
foreigners_number
foreigners_percentage
national_team_players
stadium_name
stadium_seats
net_transfer_record
coach_name
last_season
filename
url

--- competitions.csv ---
competition_id
competition_code
name
sub_type
type
country_id
country_name
domestic_league_code
confederation
total_clubs
url

--- countries.csv ---
country_id
country_name
country_code
confederation
total_clubs
total_players
average_age
url

--- game_events.csv ---
game_event_id
date
game_id
minute
type
club_id
club_name
player_id
description
player_in_id
player_assist_id

---

In [8]:
player_focused_files = [
    "players.csv",
    "appearances.csv",
    "player_valuations.csv",
    "transfers.csv",
    "game_events.csv",
    "game_lineups.csv",
]

for filename in player_focused_files:
    path = RAW_DIR / filename
    
    if path.exists():
        print(f"\n--- {filename} ---")
        df = pd.read_csv(path, nrows=10)
        print(df.shape)
        display(df.head())
    else:
        print(f"Missing: {filename}")


--- players.csv ---
(10, 26)


,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000,30000000
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000,8000000
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000,34500000
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus Football Club,200000,24500000
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,IFM,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,FC Bayern München,100000,3000000



--- appearances.csv ---
(10, 13)


,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,2,0,90
1,2233748_79232,2233748,79232,8841,2698,2012-07-05,Ruslan Abyshov,ELQ,0,0,0,0,90
2,2234413_42792,2234413,42792,6251,465,2012-07-05,Sander Puri,ELQ,0,0,0,0,45
3,2234418_73333,2234418,73333,1274,76,2012-07-05,Vegar Hedenstad,ELQ,0,0,0,0,90
4,2234421_122011,2234421,122011,195,3008,2012-07-05,Markus Henriksen,ELQ,0,0,0,1,90



--- player_valuations.csv ---
(10, 6)


...

,player_id,transfer_date,transfer_season,from_club_id,to_club_id,from_club_name,to_club_name,transfer_fee,market_value_in_eur,player_name
0,467994,2030-06-30,25/26,5621,749,Reggiana,FC Empoli,0.0,700000.0,Luca Belardinelli
1,784335,2027-07-18,27/28,6505,6502,Gimcheon Sangmu,Jeonbuk Hyundai,0.0,500000.0,Jun-soo Byeon
2,402135,2027-07-04,27/28,6505,515,Gimcheon Sangmu,Without Club,NaN,350000.0,Jun-su Ahn
3,716435,2027-07-04,27/28,6505,30925,Gimcheon Sangmu,Gwangju FC,0.0,325000.0,Kang-hyun Lee
4,803933,2027-06-30,26/27,4467,14,TSV Hartberg,Austria Vienna,0.0,300000.0,Luca Pazourek



--- game_events.csv ---
(10, 11)


,game_event_id,date,game_id,minute,type,club_id,club_name,player_id,description,player_in_id,player_assist_id
0,60d28dc455d299f5abdb65d7204fef40,2010-06-26,1026846,69,Cards,3589,South Korea,1772,"1. Yellow card , Foul",NaN,NaN
1,71280d139049475e02eb7627897da117,2010-06-26,1026846,38,Cards,3589,South Korea,27402,"1. Yellow card , Foul",NaN,NaN
2,dbb0d095a58423d8377d6b5b6b9f258f,2010-06-26,1026846,8,Goals,3449,Uruguay,44352,", Right-footed shot, 2. Tournament Goal Assist...",NaN,3408.0
3,6948606e52c10f6a2032ebe64355fe14,2010-06-26,1026846,80,Goals,3449,Uruguay,44352,", Right-footed shot, 3. Tournament Goal Assist...",NaN,72653.0
4,eb2ca121e9069fac2789c9e0be7cdbd3,2010-06-26,1026846,84,Substitutions,3449,Uruguay,44352,", Tactical",76213.0,NaN



--- game_lineups.csv ---
(10, 10)


,game_lineups_id,date,game_id,player_id,club_id,player_name,type,position,number,team_captain
0,b2dbe01c3656b06c8e23e9de714e26bb,2013-07-27,2317258,1443,610,Christian Poulsen,substitutes,Defensive Midfield,5,0
1,b50a3ec6d52fd1490aab42042ac4f738,2013-07-27,2317258,5017,610,Niklas Moisander,starting_lineup,Centre-Back,4,0
2,7d890e6d0ff8af84b065839966a0ec81,2013-07-27,2317258,9602,1090,Maarten Martens,substitutes,Left Winger,11,0
3,8c355268678b9bbc7084221b1f0fde36,2013-07-27,2317258,12282,610,Daley Blind,starting_lineup,Left-Back,17,0
4,76193074d549e5fdce4cdcbba0d66247,2013-07-27,2317258,25427,1090,Roy Beerens,starting_lineup,Right Winger,23,0


In [9]:
context_files = [
    "games.csv",
    "clubs.csv",
    "competitions.csv",
    "countries.csv",
    "national_teams.csv",
]

for filename in context_files:
    path = RAW_DIR / filename
    
    if path.exists():
        print(f"\n--- {filename} ---")
        df = pd.read_csv(path, nrows=10)
        print(df.shape)
        display(df.head())
    else:
        print(f"Missing: {filename}")


--- games.csv ---
(10, 23)


,game_id,competition_id,season,round,date,home_club_id,away_club_id,home_club_goals,away_club_goals,home_club_position,...,stadium,attendance,referee,url,home_club_formation,away_club_formation,home_club_name,away_club_name,aggregate,competition_type
0,1026846,FIWC,2009,Round of 16,2010-06-26,3449,3589,2,1,NaN,...,Nelson Mandela Bay Stadium,30597,Wolfgang Stark,https://www.transfermarkt.co.uk/spielbericht/i...,NaN,NaN,Uruguay,South Korea,2:1,national_team_competition
1,1026847,FIWC,2009,Round of 16,2010-06-27,3437,6303,3,1,NaN,...,FNB-Stadium,84377,Roberto Rosetti,https://www.transfermarkt.co.uk/spielbericht/i...,NaN,NaN,Argentina,Mexico,3:1,national_team_competition
2,1027001,FIWC,2009,Round of 16,2010-06-26,3505,3441,1,2,NaN,...,Royal Bafokeng Stadium,34976,Viktor Kassai,https://www.transfermarkt.co.uk/spielbericht/i...,NaN,NaN,United States,Ghana,1:2,national_team_competition
3,1027002,FIWC,2009,Round of 16,2010-06-27,3262,3299,4,1,NaN,...,Free State Stadium,40510,Jorge Larrionda,https://www.transfermarkt.co.uk/spielbericht/i...,NaN,NaN,Germany,England,4:1,national_team_competition
4,1027048,FIWC,2009,Round of 16,2010-06-28,3379,3503,2,1,NaN,...,Moses Mabhida Stadion,61962,Undiano Mallenco,https://www.transfermarkt.co.uk/spielbericht/i...,NaN,NaN,Netherlands,Slovakia,2:1,national_team_competition



--- clubs.csv ---
(10, 17)


,club_id,club_code,name,domestic_competition_id,total_market_value,squad_size,average_age,foreigners_number,foreigners_percentage,national_team_players,stadium_name,stadium_seats,net_transfer_record,coach_name,last_season,filename,url
0,10,arminia-bielefeld,Arminia Bielefeld,L1,NaN,27,25.3,15,55.6,4,SchücoArena,26515,+€5.90m,NaN,2021,../data/raw/transfermarkt-scraper/2021/clubs.j...,https://www.transfermarkt.co.uk/arminia-bielef...
1,10004,paris-fc,Paris Football Club,FR1,NaN,31,28.5,17,54.8,8,Stade Jean Bouin,19904,€-72.30m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/paris-fc/start...
2,10010,esporte-clube-bahia,Esporte Clube Bahia,BRA1,NaN,32,26.2,6,18.8,3,Arena Fonte Nova,47364,+€8.14m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/esporte-clube-...
3,1003,leicester-city,Leicester City,GB1,NaN,29,25.9,17,58.6,9,King Power Stadium,32259,+€57.30m,Steve Cooper,2024,../data/raw/transfermarkt-scraper/2024/clubs.j...,https://www.transfermarkt.co.uk/leicester-city...
4,1005,us-lecce,Unione Sportiva Lecce,IT1,NaN,27,25.1,23,85.2,10,Ettore Giardiniero,31559,+€8.62m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/us-lecce/start...



--- competitions.csv ---
(10, 11)


,competition_id,competition_code,name,sub_type,type,country_id,country_name,domestic_league_code,confederation,total_clubs,url
0,A1,bundesliga,bundesliga,first_tier,domestic_league,127,Austria,A1,europa,12.0,https://www.transfermarkt.co.uk/bundesliga/sta...
1,AFAC,afc-asian-cup,afc-asian-cup,afc_asian_cup,national_team_competition,-1,NaN,NaN,asien,NaN,https://www.transfermarkt.co.uk/afc-asian-cup/...
2,AFCN,africa-cup-of-nations,africa-cup-of-nations,africa_cup_of_nations,national_team_competition,-1,NaN,NaN,afrika,NaN,https://www.transfermarkt.co.uk/africa-cup-of-...
3,ARG1,torneo-apertura,torneo-apertura,first_tier,domestic_league,9,Argentina,ARG1,amerika,30.0,https://www.transfermarkt.co.uk/torneo-apertur...
4,AUS1,a-league-men,a-league-men,first_tier,domestic_league,12,Australia,AUS1,asien,12.0,https://www.transfermarkt.co.uk/a-league-men/s...



--- countries.csv ---
(10, 8)


,country_id,country_name,country_code,confederation,total_clubs,total_players,average_age,url
0,10,Armenia,ARM1,europa,10,265,24.8,https://www.transfermarkt.co.uk/wettbewerbe/na...
1,100,North Macedonia,MAZ1,europa,12,283,25.4,https://www.transfermarkt.co.uk/wettbewerbe/na...
2,103,Malaysia,MYS1,asien,13,399,28.3,https://www.transfermarkt.co.uk/wettbewerbe/na...
3,106,Malta,MT1N,europa,12,299,26.9,https://www.transfermarkt.co.uk/wettbewerbe/na...
4,107,Morocco,MAR1,afrika,16,499,27.2,https://www.transfermarkt.co.uk/wettbewerbe/na...



--- national_teams.csv ---
(10, 17)


,national_team_id,name,team_code,country_id,country_name,country_code,confederation,team_image_url,squad_size,average_age,foreigners_number,foreigners_percentage,total_market_value,coach_name,fifa_ranking,last_season,url
0,10521,San Marino,san-marino,144,San Marino,SMR1,UEFA,https://tmssl.akamaized.net//images/flagge/hea...,24,24.6,19,79.2,935000,NaN,210,2025,https://www.transfermarkt.co.uk/san-marino/sta...
1,10533,Andorra,andorra,5,Andorra,AND2,UEFA,https://tmssl.akamaized.net//images/flagge/hea...,28,28.4,16,57.1,3520000,NaN,172,2025,https://www.transfermarkt.co.uk/andorra/starts...
2,11953,Montenegro,montenegro,216,Montenegro,MNE1,UEFA,https://tmssl.akamaized.net//images/flagge/hea...,24,26.4,23,95.8,58130001,NaN,82,2025,https://www.transfermarkt.co.uk/montenegro/sta...
3,13342,Guatemala,guatemala,58,Guatemala,GU1A,CONCACAF,https://tmssl.akamaized.net//images/flagge/hea...,21,25.1,6,28.6,5380000,NaN,94,2025,https://www.transfermarkt.co.uk/guatemala/star...
4,13497,Uganda,uganda,176,Uganda,UGL1,CAF,https://tmssl.akamaized.net//images/flagge/hea...,28,27.9,23,82.1,8630000,NaN,88,2025,https://www.transfermarkt.co.uk/uganda/startse...


In [10]:
missing_value_summaries = {}

for path in csv_files:
    size_mb = path.stat().st_size / 1024 / 1024
    
    if size_mb <= 100:
        df = pd.read_csv(path)
        missing_value_summaries[path.name] = df.isna().sum().sort_values(ascending=False).head(20)
    else:
        print(f"Skipped {path.name}: {size_mb:.2f} MB")

missing_value_summaries

Skipped appearances.csv: 139.77 MB
Skipped game_events.csv: 145.83 MB
Skipped game_lineups.csv: 322.34 MB


{'club_games.csv': own_position             50444
 opponent_position        50444
 own_manager_name          1688
 opponent_manager_name     1688
 game_id                      0
 club_id                      0
 own_goals                    0
 opponent_id                  0
 opponent_goals               0
 hosting                      0
 is_win                       0
 dtype: int64,
 'clubs.csv': total_market_value         796
 coach_name                 706
 foreigners_percentage       57
 average_age                 38
 stadium_name                 0
 filename                     0
 last_season                  0
 net_transfer_record          0
 stadium_seats                0
 club_id                      0
 national_team_players        0
 club_code                    0
 foreigners_number            0
 squad_size                   0
 domestic_competition_id      0
 name                         0
 url                          0
 dtype: int64,
 'competitions.csv': total_clubs           

In [11]:
metadata = {
    "provider": "Kaggle",
    "dataset": "davidcariboo/player-scores",
    "source_url": "https://www.kaggle.com/datasets/davidcariboo/player-scores",
    "underlying_source": "Transfermarkt",
    "source_type": "multi-file CSV dataset",
    "downloaded_or_inventory_created_at_utc": datetime.now(timezone.utc).isoformat(),
    "local_raw_folder": str(RAW_DIR),
    "files": inventory_df[["file", "rows", "columns", "size_mb", "column_names"]].to_dict(orient="records"),
    "notes": [
        "Local raw dataset inventory.",
        "Raw CSV files are not committed to Git.",
        "No cleaning, feature engineering, SQL import, or modeling performed.",
        "Dataset is player-focused but also includes match, club, competition, and national team context tables."
    ],
}

metadata_path = RAW_DIR / "kaggle_player_scores_inventory_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

metadata_path

PosixPath('../data/raw/kaggle_player_scores/kaggle_player_scores_inventory_metadata.json')